# EDA inicial para el proyecto de nanotoxicidad

Esta notebook prepara la exploración inicial del dataset disponible en el repositorio para construir el proyecto integrador de la Unidad 6.

Objetivos de esta notebook:
- identificar el dataset más útil para arrancar
- entender qué representa cada variable
- decidir el tipo de problema de Machine Learning
- evaluar si el dataset es suficiente para un proyecto universitario de 3 semanas
- dejar una base lista para conectar con `U6_03_IMPLEMENTACION_PROYECTO.ipynb`

## 1. Dataset recomendado para comenzar

El archivo más útil dentro del repositorio para iniciar el trabajo es `educational_content/unit_03_ml_nanomaterials/nanomaterials_full_dataset.csv`.

**Por qué conviene para empezar**
- Ya existe dentro del repositorio.
- Tiene variables numéricas y categóricas útiles para un pipeline real.
- Es pequeño y manejable para un proyecto universitario de 3 semanas.
- Permite construir EDA, limpieza, pipeline y modelo sin depender de una descarga externa inmediata.

**Limitación importante**
- Este dataset describe propiedades de nanomateriales, pero no tiene una etiqueta explícita de toxicidad.
- Por eso, para el modelo final de nanotoxicidad se debe complementar con una fuente de labels de toxicidad o construir una etiqueta derivada y justificable.

**Conclusión práctica**
- Sirve como base estructural y de pipeline.
- No es suficiente por sí solo para un clasificador final de toxicidad sin una variable objetivo adecuada.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

dataset_path = Path("..") / "unit_03_ml_nanomaterials" / "nanomaterials_full_dataset.csv"
if not dataset_path.exists():
    dataset_path = Path("c:/Users/natal/OneDrive/Documentos/PROYECTO IA/Antigravity-Nano-Research-Multiagentic-Core/educational_content/unit_03_ml_nanomaterials/nanomaterials_full_dataset.csv")

df = pd.read_csv(dataset_path)
print(f'Archivo cargado: {dataset_path}')
print(f'Forma del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

## 2. ¿Qué representa cada variable?

**Variables categóricas principales**
- `element`: elemento químico base de la nanopartícula.
- `geometry`: geometría o forma estructural del nanoclúster.
- `element_group`: grupo químico del elemento.

**Variables estructurales / geométricas**
- `n_atoms`: número de átomos.
- `noshells`: número de capas o shells.
- `radius_mean`, `radius_std`, `radius_max`: resumen estadístico del radio estructural.
- `asphericity`: qué tan alejada está la partícula de una forma esférica.
- `compactness`: compacidad geométrica.
- `surface_fraction`: fracción superficial estimada.
- `coordination_mean`, `coordination_std`, `coordination_min`, `coordination_max`: estadística de coordinación atómica.

**Variables energéticas / físicas**
- `energy_per_atom`: energía por átomo.
- `energy_total`: energía total.
- `energy_stability`: indicador de estabilidad energética.
- `melting_point`: punto de fusión estimado.
- `log_n_atoms`: logaritmo del número de átomos.

**Interpretación para nanotoxicidad**
Estas variables no describen toxicidad directamente, pero sí capturan rasgos que luego pueden relacionarse con toxicidad: tamaño, forma, estabilidad, coordinación y superficie.

In [ ]:
# Inspección rápida del esquema del dataset
print('Columnas:')
for col in df.columns:
    print('-', col)

print('\nTipos de dato:')
print(df.dtypes)

print('\nValores faltantes por columna:')
print(df.isna().sum().sort_values(ascending=False))

print('\nFilas duplicadas:', df.duplicated().sum())

## 3. Variable objetivo (target)

Para el proyecto final de nanotoxicidad, la variable objetivo ideal debería ser una de estas dos opciones:

1. **Clasificación binaria**: `toxico` / `no_toxico`.
2. **Regresión**: un `toxicity_score` continuo entre 0 y 1, o una escala experimental equivalente.

**Decisión recomendada para comenzar**
- Si el dataset de toxicidad aún no está consolidado, conviene diseñar el proyecto como **clasificación binaria**.
- Razón: es más defendible con datos limitados, más fácil de explicar y más compatible con un MVP universitario de 3 semanas.

**Importante**
- El archivo actual no trae una etiqueta de toxicidad directa.
- Por eso, esta notebook se usa para preparar la estructura y el pipeline, no para entrenar todavía el clasificador final.

In [ ]:
# Perfil básico del dataset
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print('Columnas numéricas:', numeric_cols)
print('Columnas categóricas:', categorical_cols)

display(df[numeric_cols].describe().T)

for col in categorical_cols:
    
    print(f'\nFrecuencias de {col}:')
    print(df[col].value_counts(dropna=False).head(10))

## 4. ¿Conviene clasificación binaria o regresión?

**Recomendación**: comenzar con **clasificación binaria**.

**Motivos**
- La toxicidad suele presentarse como una decisión de riesgo: pasa / no pasa.
- Es más fácil conseguir o derivar etiquetas binarias confiables.
- Las métricas son claras: accuracy, precision, recall, F1 y ROC-AUC.
- El resultado es fácil de explicar en presentación y reporte.

**Cuándo usar regresión**
- Solo si consigues un índice de toxicidad continuo bien definido.
- Si las etiquetas son débiles o poco consistentes, la regresión puede introducir más ruido que valor.

In [ ]:
# Matriz de correlación para variables numéricas
plt.figure(figsize=(12, 8))
corr = df[numeric_cols].corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Correlación entre variables numéricas')
plt.tight_layout()
plt.show()

## 5. ¿El dataset es suficiente para un proyecto universitario de 3 semanas?

**Sí, como base de trabajo y prototipo.**

**Pero con una condición importante**
- Este dataset por sí solo no alcanza para un modelo final de nanotoxicidad si no incluye la etiqueta objetivo.
- Lo suficiente aquí es la **estructura**, no el target.

**Evaluación práctica**
- Tamaño: pequeño, manejable y rápido de iterar.
- Complejidad: adecuada para el tiempo disponible.
- Viabilidad académica: alta, si se complementa con una fuente de labels de toxicidad o una estrategia de etiquetado justificable.

**Veredicto**
- Adecuado para arrancar.
- No suficiente como única fuente final de entrenamiento para toxicidad.

## 6. Preparación del dataset para el pipeline de U6

El siguiente paso para U6_03 será: 

1. Definir columnas de entrada.
2. Separar variables numéricas y categóricas.
3. Crear un preprocesamiento con `ColumnTransformer`.
4. Construir un `Pipeline` con imputación, escalado y modelo.
5. Sustituir este dataset estructural por uno con target de toxicidad cuando esté listo.

La idea es que esta notebook deje lista la capa de exploración y preparación antes de entrenar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Ejemplo de preparación de pipeline, listo para U6_03
feature_cols = [c for c in df.columns if c not in []]
X = df.copy()

numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

print('Preprocesador listo para integrarse a U6_03.')

## 7. Visualizaciones iniciales

Estas figuras sirven como evidencia para el reporte y como primer acercamiento al dataset.

Sugerencias de lectura:
- distribuciones de tamaño y energía
- relaciones entre estabilidad y coordinación
- comparación por elemento o geometría

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

if 'n_atoms' in df.columns:
    sns.histplot(df['n_atoms'], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title('Distribución de n_atoms')

if 'energy_per_atom' in df.columns:
    sns.histplot(df['energy_per_atom'], kde=True, ax=axes[1], color='darkorange')
    axes[1].set_title('Distribución de energy_per_atom')

if 'coordination_mean' in df.columns and 'energy_stability' in df.columns:
    sns.scatterplot(data=df, x='coordination_mean', y='energy_stability', ax=axes[2], color='forestgreen')
    axes[2].set_title('Coordinación media vs estabilidad energética')

if 'geometry' in df.columns:
    order = df['geometry'].value_counts().index
    sns.countplot(data=df, y='geometry', order=order, ax=axes[3], color='slateblue')
    axes[3].set_title('Frecuencia por geometría')

plt.tight_layout()
plt.show()

## 8. Recomendación final para continuar

Para seguir con el proyecto de nanotoxicidad de forma simple y defendible:

1. Conserva este dataset como base estructural.
2. Agrega una fuente de etiqueta de toxicidad.
3. Define el problema como clasificación binaria.
4. Lleva el preprocesamiento a `U6_03_IMPLEMENTACION_PROYECTO.ipynb`.
5. Integra el safety gate con `external_skills.ai_mining.toxicity_predictor`.
6. Despliega el mejor modelo con la API de `mi_proyecto_api/`.

Con esto ya tienes una base compatible con la arquitectura de la Unidad 6.